In [2]:
from sqlalchemy import text
from sqlalchemy.ext.asyncio import create_async_engine, AsyncSession, async_sessionmaker
from hyrule_football.config import settings
from hyrule_football.utils import get_logger
from hyrule_football.database import Base  # ✅ 使用项目的 Base，而不是创建新的

logger = get_logger(__name__)

async_engine = create_async_engine(
    settings.MYSQL_DB_URL,
    echo=True,  # 开发时显示 SQL，生产环境改为 False
    pool_pre_ping=True,  # 连接池健康检查
)

# 异步会话工厂
AsyncSessionLocal = async_sessionmaker(
    async_engine,
    class_=AsyncSession,
    expire_on_commit=False,
    autocommit=False,
    autoflush=False,
)

async def create_db_if_need():
    """创建 MySQL 数据库（如果不存在）"""
    db_name = settings.MYSQL_DB_URL.split("/")[-1]
    base_url = settings.MYSQL_DB_URL.rsplit("/", 1)[0]

    tmp_engine = create_async_engine(base_url, isolation_level="AUTOCOMMIT")
    
    async with tmp_engine.connect() as conn:
        # 创建数据库
        await conn.execute(text(
            f"CREATE DATABASE IF NOT EXISTS {db_name} "
            "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"
        ))

        logger.info(f"✅ 数据库 {db_name} 创建成功（或已存在）")
    
    await tmp_engine.dispose()


async def init_db_tables():
    """创建 MySQL 所有表（异步）"""
    async with async_engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)

    logger.info("✅ MySQL 所有表创建成功")


async def init_database():
    """完整初始化 MySQL：创建数据库 + 创建表"""
    await create_db_if_need()
    await init_db_tables()



In [3]:
from hyrule_football.models import Company
await init_database()

Can't create database 'hyrule_football_db'; database exists
2025-12-20 00:05:09 - __main__ - INFO - ✅ 数据库 hyrule_football_db 创建成功（或已存在）


2025-12-20 00:05:09,361 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2025-12-20 00:05:09,362 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-12-20 00:05:09,365 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2025-12-20 00:05:09,366 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-12-20 00:05:09,368 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2025-12-20 00:05:09,369 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-12-20 00:05:09,373 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-12-20 00:05:09,374 INFO sqlalchemy.engine.Engine DESCRIBE `hyrule_football_db`.`company`
2025-12-20 00:05:09,375 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-12-20 00:05:09,396 INFO sqlalchemy.engine.Engine 
CREATE TABLE company (
	id INTEGER NOT NULL COMMENT '博彩公司ID' AUTO_INCREMENT, 
	name VARCHAR(100) NOT NULL COMMENT '博彩公司名称', 
	created_at DATETIME COMMENT '创建时间', 
	updated_at DATETIME COMMENT '更新时间', 
	PRIMARY KEY (id)
)COMMENT='博彩公司表'


2025-12-20 00:05:09,397 INFO sqlalchemy.engi

2025-12-20 00:05:09 - __main__ - INFO - ✅ MySQL 所有表创建成功
